# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (History & Move Ordering Phase 4)

本 Notebook 用於執行 **Phase 4：歷史表反饋與走步排序啟發式體系（History Heuristics & Move Ordering）** 雲端 SPSA 調參與深度 SPRT 對抗驗收。

### 🎯 Phase 4 核心調參六軸
1. `LMR_HISTORY_SCALE`（當前基線 204）：LMR 歷史衰減係數，控制歷史評分對剪枝深度的微調力道。
2. `HISTORY_BONUS_SCALE`（當前基線 120）：安靜步剪枝歷史加分斜率（`bonus = min(SCALE * depth, CAP)`）。
3. `HISTORY_BONUS_CAP`（當前基線 1800）：安靜步歷史加分上限。
4. `HISTORY_MALUS_CAP`（當前基線 1600）：失敗走步歷史懲罰扣分上限。
5. `QUIET_ORDER_KILLER_1`（當前基線 400000）：第一殺手步（Killer Move 1）安靜步排序加分。
6. `QUIET_ORDER_COUNTER`（當前基線 300000）：應對步（Counter Move）安靜步排序加分。

### ⚡ 調參架構
* **L1 雲端 SPSA 探索**：80 輪 × 150 局 = 12,000 局 @ 50,000 nodes（`inprocess` 免重複 JIT 高速對抗）
* **Polyak-Ruppert 平滑取樣**：實證黃金比例 **`--tail 0.3`**（後 24 輪 / 3,600 局加權平均）
* **L2 深度獨立驗收**：600 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)


In [ ]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version


In [ ]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!rm -f tournament_analysis/l2_verify_history_p4_deep_tail03.json
!git pull origin {branch}
!pip install -q numba chess numpy


In [ ]:
# 3. 測試環境與驗證 Phase 4 基準常數
import sys, json
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase4_params = [
    "LMR_HISTORY_SCALE",
    "HISTORY_BONUS_SCALE",
    "HISTORY_BONUS_CAP",
    "HISTORY_MALUS_CAP",
    "QUIET_ORDER_KILLER_1",
    "QUIET_ORDER_COUNTER"
]

print("🔍 Phase 4 歷史表與排序啟發式六軸基準值（Baseline）：")
print(json.dumps(get_default_params(phase4_params), indent=2))


In [ ]:
# 4. 啟動 L1 SPSA Phase 4 歷史與排序六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/history_p4_deep_l1.jsonl"
state_file = "tune_search/states/history_p4_deep_l1.json"
total_campaign_iters = 80

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params LMR_HISTORY_SCALE,HISTORY_BONUS_SCALE,HISTORY_BONUS_CAP,HISTORY_MALUS_CAP,QUIET_ORDER_KILLER_1,QUIET_ORDER_COUNTER \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 4 的 80 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")


In [ ]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_p4_deep.json
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_history_p4_tail03_uniform.json
!python tune_search/summarize_run.py tune_search/logs/history_p4_deep_l1.jsonl --tail 0.3 --weight-mode linear --out tune_search/candidate_history_p4_tail03_linear.json
print("\n=== Phase 4 等權最佳候選參數 JSON (tail 0.3 uniform) ===")
!cat tune_search/candidate_history_p4_deep.json
print("\n=== Phase 4 線性加權候選參數 JSON (tail 0.3 linear) ===")
!cat tune_search/candidate_history_p4_tail03_linear.json


In [ ]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為 Phase 4 Baseline (tune_search/baseline_history_p4.json)
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 Phase 4 L2 深度驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_history_p4_deep.json \
  --baseline-json tune_search/baseline_history_p4.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_history_p4_deep_tail03.json
